# Data Quality Reporting with Claude API

This project demonstrates three core capabilities of the Claude API applied to an HR Employee Attrition dataset:

1. **Data Cleaning & Validation** — Structured JSON output via assistant message prefilling and stop sequences
2. **Natural Language to SQL** — Prompt engineering with XML tag structuring
3. **Automated Report Generation** — System prompts with multi-turn message formatting and context handling

**Dataset:** [IBM HR Analytics Employee Attrition & Performance](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset) (1,470 employees, 35 features)

## 1. Setup

In [ ]:
# Install dependencies
%pip install anthropic python-dotenv pandas

In [2]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from anthropic import Anthropic

# Load API key from .env file
load_dotenv()

# Initialize the Anthropic client and select model
client = Anthropic()
model = "claude-sonnet-4-5"

## 2. Helper Functions

These utility functions manage the conversation message list and send requests to the Claude API.
The `chat` function uses `**kwargs` so it can forward any additional API parameters
(e.g. `stop_sequences`, `temperature`) without needing to define each one explicitly.

In [3]:
def add_user_message(messages, content):
    """Append a user message to the conversation history."""
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, content):
    """Append an assistant message to the conversation history.
    Used for prefilling — tells Claude it has already started responding
    with this text, so it continues from here."""
    messages.append({"role": "assistant", "content": content})


def chat(messages, system=None, **kwargs):
    """Send messages to Claude and return the response text.

    Parameters
    ----------
    messages : list  — conversation history
    system   : str   — optional system prompt (defines Claude's role)
    **kwargs         — forwarded to the API (stop_sequences, temperature, etc.)
    """
    params = {
        "model": model,
        "max_tokens": 4096,
        "messages": messages,
    }
    if system:
        params["system"] = system

    message = client.messages.create(**params, **kwargs)
    return message.content[0].text


def chat_stream(messages, system=None, **kwargs):
    """Stream Claude's response token by token and return the full text.
    Provides a real-time typing effect instead of waiting for the entire response."""
    params = {
        "model": model,
        "max_tokens": 4096,
        "messages": messages,
    }
    if system:
        params["system"] = system

    with client.messages.stream(**params, **kwargs) as stream:
        full_text = ""
        for text in stream.text_stream:
            print(text, end="", flush=True)
            full_text += text
    print()  # newline after streaming finishes
    return full_text

## 3. Load and Explore the Dataset

Load the IBM HR Attrition dataset and inspect its structure.

In [4]:
df = pd.read_csv("HR-Employee-Attrition.csv")

print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")
print(f"\nColumn names:\n{list(df.columns)}")
print(f"\nAttrition distribution:")
print(df["Attrition"].value_counts())
df.head()

Rows: 1470
Columns: 35

Column names:
['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

Attrition distribution:
Attrition
No     1233
Yes     237
Name: count, dtype: int64


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


---
## Module 1: Automated Data Cleaning & Validation

### Approach
1. Take a small subset of rows and **introduce realistic data quality issues** (typos, missing values, inconsistent formatting, invalid entries, duplicates)
2. Send the dirty data to Claude inside XML tags with clear cleaning rules
3. Use **assistant message prefilling** (`"```json\n"`) and **stop sequences** (`["```"]`) to force Claude to return only valid JSON
4. Parse the JSON response and display the cleaned data

### Step 1a: Create a dirty sample from the real dataset

In [5]:
# Take 12 rows and introduce realistic data quality issues
sample = df.head(12).copy()

# Issue 1: Inconsistent department names (typos and casing)
sample.loc[0, "Department"] = "sales"             # should be "Sales"
sample.loc[3, "Department"] = "Reserch & Development"  # typo
sample.loc[7, "Department"] = "HUMAN RESOURCES"    # wrong casing

# Issue 2: Missing values
sample.loc[2, "JobRole"] = None
sample.loc[9, "MonthlyIncome"] = None

# Issue 3: Invalid values
sample.loc[4, "Age"] = -29                         # negative age
sample.loc[6, "MonthlyIncome"] = -5000             # negative income

# Issue 4: Duplicate row
duplicate_row = sample.iloc[1:2].copy()
sample = pd.concat([sample, duplicate_row], ignore_index=True)

# Issue 5: Inconsistent attrition values
sample.loc[5, "Attrition"] = "yes"                 # should be "Yes"
sample.loc[8, "Attrition"] = "NO"                  # should be "No"

# Select relevant columns to keep the prompt concise
cols = ["EmployeeNumber", "Age", "Department", "JobRole",
        "MonthlyIncome", "Attrition", "YearsAtCompany", "OverTime"]
dirty_df = sample[cols].copy()

print(f"Dirty sample: {len(dirty_df)} rows, {len(cols)} columns")
dirty_df

Dirty sample: 13 rows, 8 columns


,EmployeeNumber,Age,Department,JobRole,MonthlyIncome,Attrition,YearsAtCompany,OverTime
0,1,41,sales,Sales Executive,5993.0,Yes,6,Yes
1,2,49,Research & Development,Research Scientist,5130.0,No,10,No
2,4,37,Research & Development,NaN,2090.0,Yes,0,Yes
3,5,33,Reserch & Development,Research Scientist,2909.0,No,8,Yes
4,7,-29,Research & Development,Laboratory Technician,3468.0,No,2,No
5,8,32,Research & Development,Laboratory Technician,3068.0,yes,7,No
6,10,59,Research & Development,Laboratory Technician,-5000.0,No,1,Yes
7,11,30,HUMAN RESOURCES,Laboratory Technician,2693.0,No,1,No
8,12,38,Research & Development,Manufacturing Director,9526.0,NO,9,No
9,13,36,Research & Development,Healthcare Representative,NaN,No,7,No


### Step 1b: Send dirty data to Claude for cleaning

In [6]:
def clean_data(csv_text):
    """Send raw CSV to Claude for cleaning.
    Uses prefilling and stop sequences to guarantee JSON output."""
    messages = []

    prompt = f"""<task>
Clean and validate the following HR employee CSV data.
</task>

<data>
{csv_text}
</data>

<rules>
- Standardize Department names to title case: "Sales", "Research & Development", "Human Resources"
- Standardize Attrition values to "Yes" or "No"
- Flag rows with negative Age or negative MonthlyIncome as invalid
- Flag rows with missing JobRole or missing MonthlyIncome
- Remove duplicate rows based on EmployeeNumber, keep the first occurrence
- Do not invent or fill in missing values, just flag them
</rules>

<output_format>
Return a JSON object with exactly two keys:
- "cleaned_records": array of objects with corrected fields (exclude flagged invalid rows)
- "issues_found": array of strings, each describing one issue that was fixed or flagged
</output_format>"""

    add_user_message(messages, prompt)

    # Prefill: tell Claude it has already started a JSON code block
    add_assistant_message(messages, "```json")

    # Stop sequence: cut output when the code block closes
    text = chat(messages, stop_sequences=["```"], temperature=0)

    return json.loads(text.strip())

In [7]:
# Convert dirty dataframe to CSV text and clean it
dirty_csv = dirty_df.to_csv(index=False)

result = clean_data(dirty_csv)

# Display issues found
print("=" * 60)
print("ISSUES FOUND")
print("=" * 60)
for i, issue in enumerate(result["issues_found"], 1):
    print(f"  {i}. {issue}")

# Display cleaned records
print(f"\n{'=' * 60}")
print(f"CLEANED RECORDS ({len(result['cleaned_records'])} rows)")
print("=" * 60)
cleaned_df = pd.DataFrame(result["cleaned_records"])
cleaned_df

ISSUES FOUND
  1. Row 1 (Employee 1): Standardized Department from 'sales' to 'Sales'
  2. Row 4 (Employee 4): Missing JobRole - row flagged as invalid
  3. Row 5 (Employee 5): Standardized Department from 'Reserch & Development' to 'Research & Development'
  4. Row 7 (Employee 7): Negative Age (-29) - row flagged as invalid
  5. Row 8 (Employee 8): Standardized Attrition from 'yes' to 'Yes'
  6. Row 10 (Employee 10): Negative MonthlyIncome (-5000.0) - row flagged as invalid
  7. Row 11 (Employee 11): Standardized Department from 'HUMAN RESOURCES' to 'Human Resources'
  8. Row 12 (Employee 12): Standardized Attrition from 'NO' to 'No'
  9. Row 13 (Employee 13): Missing MonthlyIncome - row flagged as invalid
  10. Row 14 (Employee 2): Duplicate EmployeeNumber - row removed (kept first occurrence)

CLEANED RECORDS (8 rows)


,EmployeeNumber,Age,Department,JobRole,MonthlyIncome,Attrition,YearsAtCompany,OverTime
0,1,41,Sales,Sales Executive,5993.0,Yes,6,Yes
1,2,49,Research & Development,Research Scientist,5130.0,No,10,No
2,5,33,Research & Development,Research Scientist,2909.0,No,8,Yes
3,8,32,Research & Development,Laboratory Technician,3068.0,Yes,7,No
4,11,30,Human Resources,Laboratory Technician,2693.0,No,1,No
5,12,38,Research & Development,Manufacturing Director,9526.0,No,9,No
6,14,35,Research & Development,Laboratory Technician,2426.0,No,5,No
7,15,29,Research & Development,Laboratory Technician,4193.0,No,9,Yes


---
## Module 2: Natural Language to SQL Conversion

### Approach
1. Define the table schema so Claude knows the column names and types
2. Wrap the schema and user question in **XML tags** to keep them clearly separated
3. Use **prefilling** with `"```sql\n"` to force SQL-only output
4. Use **stop sequences** to prevent any commentary after the query

In [8]:
# Define the table schema based on the actual dataset
TABLE_SCHEMA = """
Table: hr_employees
Columns:
  - EmployeeNumber   INTEGER   (primary key)
  - Age              INTEGER
  - Attrition        TEXT      ('Yes' or 'No')
  - BusinessTravel   TEXT      ('Travel_Rarely', 'Travel_Frequently', 'Non-Travel')
  - Department        TEXT      ('Sales', 'Research & Development', 'Human Resources')
  - DistanceFromHome  INTEGER
  - Education         INTEGER   (1-5 scale)
  - EducationField    TEXT
  - Gender            TEXT      ('Male', 'Female')
  - JobLevel          INTEGER   (1-5)
  - JobRole           TEXT
  - JobSatisfaction   INTEGER   (1-4 scale)
  - MaritalStatus     TEXT      ('Single', 'Married', 'Divorced')
  - MonthlyIncome     INTEGER
  - NumCompaniesWorked INTEGER
  - OverTime          TEXT      ('Yes', 'No')
  - PercentSalaryHike  INTEGER
  - PerformanceRating  INTEGER  (1-4)
  - TotalWorkingYears  INTEGER
  - YearsAtCompany     INTEGER
  - YearsInCurrentRole INTEGER
  - YearsSinceLastPromotion INTEGER
"""

print("Table schema defined for hr_employees")
print(f"Columns: 22")

Table schema defined for hr_employees
Columns: 22


In [9]:
def nl_to_sql(question):
    """Convert a natural language question to a SQL query.
    Uses XML tags to structure the prompt and prefilling for clean output."""
    messages = []

    prompt = f"""<task>
Convert the following natural language question into a SQL query.
</task>

<table_schema>
{TABLE_SCHEMA}
</table_schema>

<question>
{question}
</question>

<rules>
- Write standard SQL compatible with SQLite
- Use descriptive column aliases with AS
- Return only the SQL query, no explanation
- Use single quotes for string literals
</rules>"""

    add_user_message(messages, prompt)

    # Prefill with sql code block
    add_assistant_message(messages, "```sql")

    text = chat(messages, stop_sequences=["```"], temperature=0)
    return text.strip()

In [10]:
# Test with multiple questions of increasing complexity
questions = [
    "What is the average monthly income by department?",
    "Which job roles have an attrition rate above 20%?",
    "How many employees work overtime in each department, and what percentage is that?",
    "Find the top 5 job roles with the highest average years at company among employees who left",
]

for i, q in enumerate(questions, 1):
    sql = nl_to_sql(q)
    print(f"Question {i}: {q}")
    print(f"\n{sql}")
    print("\n" + "=" * 60 + "\n")

Question 1: What is the average monthly income by department?

SELECT 
    Department,
    AVG(MonthlyIncome) AS AverageMonthlyIncome
FROM 
    hr_employees
GROUP BY 
    Department


Question 2: Which job roles have an attrition rate above 20%?

SELECT 
  JobRole,
  ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS AttritionRate
FROM hr_employees
GROUP BY JobRole
HAVING 100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) > 20
ORDER BY AttritionRate DESC


Question 3: How many employees work overtime in each department, and what percentage is that?

SELECT 
  Department,
  SUM(CASE WHEN OverTime = 'Yes' THEN 1 ELSE 0 END) AS EmployeesWorkingOvertime,
  ROUND(100.0 * SUM(CASE WHEN OverTime = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS PercentageWorkingOvertime
FROM hr_employees
GROUP BY Department
ORDER BY Department


Question 4: Find the top 5 job roles with the highest average years at company among employees who left

SELECT 
  JobRo

---
## Module 3: Automated Analysis Report Generation

### Approach
1. Use a **system prompt** to set Claude's role as a senior HR data analyst
2. Send the full dataset directly to Claude inside XML tags — let Claude do the analysis itself
3. Use **multi-turn conversation**: first send the data, then request a focused report
4. Use **response streaming** to display the report as it generates in real time

This demonstrates **message formatting** (XML tags to structure the data),
**context handling** (multi-turn conversation maintains data across turns),
and **system prompts** (defining Claude's role and output format).

In [11]:
# System prompt defines Claude's role and report structure
ANALYST_SYSTEM_PROMPT = """
You are a senior HR data analyst.
You analyze raw data and write clear, data-driven reports.
Always calculate specific numbers and percentages from the data provided.
Structure every report with these sections:
1. Executive Summary (2-3 sentences)
2. Key Findings (3-4 findings with specific numbers)
3. Risk Factors (which employee segments have the highest attrition risk)
4. Recommendations (2-3 actionable suggestions based on the data)
Keep the total report under 500 words.
"""

### Turn 1: Send the full dataset to Claude

In [12]:
# Full dataset has 1,470 rows — sample 500 to stay within API rate limits
sample_df = df.head(500)
full_csv = sample_df.to_csv(index=False)

messages = []

add_user_message(messages, f"""<dataset>
{full_csv}
</dataset>

<dataset_info>
This is the IBM HR Analytics Employee Attrition dataset.
This sample contains 500 records from the full {len(df)} employee dataset with {len(df.columns)} features including
demographics, job details, satisfaction scores, and whether the employee left (Attrition).
</dataset_info>

Please confirm you received the data and tell me the overall attrition rate
you calculated from the dataset.""")

context_response = chat(messages, system=ANALYST_SYSTEM_PROMPT)
add_assistant_message(messages, context_response)

print("Turn 1 — Claude confirms the data:")
print(context_response)
print("\n" + "=" * 60)

Turn 1 — Claude confirms the data:
I've received the IBM HR Analytics Employee Attrition dataset with 500 employee records.

**Calculated Overall Attrition Rate:**

From the 500 employees in this dataset:
- **Employees who left (Attrition = "Yes"):** 83
- **Employees who stayed (Attrition = "No"):** 417

**Overall Attrition Rate: 16.6%**

This means that approximately 1 in 6 employees in this sample have left the organization. This is a significant attrition rate that warrants detailed analysis.

I'm ready to analyze this data and provide you with a comprehensive HR data report identifying key patterns, risk factors, and actionable recommendations. Would you like me to proceed with the full analysis?



### Turn 2: Request the analysis report (streamed)

In [13]:
# Ask Claude to analyze the data and generate a report
add_user_message(messages, """<analysis_focus>
Analyze attrition risk factors in this dataset.
Focus on: overtime, monthly income, job satisfaction, years at company, and department.
Calculate the actual numbers from the data I sent you.
</analysis_focus>

Generate a structured HR analysis report following the format in your instructions.
Base everything on the actual data — do not make up numbers.""")

print("Turn 2 — Analysis Report (streamed):\n")
print("=" * 60)
report = chat_stream(messages, system=ANALYST_SYSTEM_PROMPT)
print("=" * 60)

Turn 2 — Analysis Report (streamed):

# HR Analytics Report: Employee Attrition Risk Factors

## Executive Summary

Analysis of 500 employee records reveals a 16.6% overall attrition rate (83 departures). Overtime emerges as the most critical risk factor, with employees working overtime showing 31.3% attrition versus 10.4% for non-overtime workers—a 3x higher risk. Income disparities are stark: departed employees earned 38% less on average ($4,787 vs $6,833 monthly). The first year represents peak flight risk at 28.6% attrition.

## Key Findings

**1. Overtime Impact (Most Critical Factor)**
- Employees working overtime: 31.3% attrition rate (48 out of 153 left)
- Employees not working overtime: 10.1% attrition rate (35 out of 347 left)
- Overtime workers are 3.1x more likely to leave the company

**2. Income Disparity**
- Average monthly income of employees who left: $4,787
- Average monthly income of employees who stayed: $6,833
- Gap: $2,046 (38% lower income for those who departed)

---
## Summary of API Techniques Used

| Module | Technique | API Feature | Purpose |
|--------|-----------|-------------|---------|
| Data Cleaning | Prefill + Stop Sequences | `add_assistant_message` + `stop_sequences=["```"]` | Force pure JSON output |
| NL to SQL | XML Tags + Prefill | `<table_schema>`, `<question>`, `<rules>` | Separate schema from question |
| Report Generation | System Prompt + Multi-turn + Streaming | `system=`, `chat_stream()` | Role definition and context handling |

**Additional parameters used:**
- `temperature=0` for deterministic output (data cleaning and SQL)
- `**kwargs` pattern for flexible API parameter forwarding
- `max_tokens` to control response length